# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR<sup>2</sup> dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Install mlcroissant if not already present
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name', '<No name>')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets by their @id, fields, and columns

# Retrieve all record set objects
record_sets = list(dataset.record_sets)

print(f"There are {len(record_sets)} record set(s) in the dataset.")
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    print(f"  Name: {rs.get('name', '<no name>')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):  # If only one field present
        fields = [fields]
    if fields:
        print(f"  Fields and their @id:")
        for field in fields:
            if isinstance(field, dict):
                print(f"    • {field.get('@id', '<no @id>')}  \t(name: {field.get('name', '<no name>')})")
            else:
                print(f"    • {field}")

## 3. Data Extraction
Load data from the record set(s) into pandas DataFrames for analysis. Each field is referenced by its `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for Record Set '@id': {record_set_id}")
    print(f"Columns (@id): {list(df.columns)}\n")

# Preview first few rows of the first record set (if available)
if record_set_ids:
    print(f"First 5 records for record set {record_set_ids[0]}")
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
We will perform EDA on a numeric field in the main record set, including filtering, normalization, and group-wise aggregation. 

All fields, columns, or groups referenced will use their Croissant `@id`.

In [ ]:
# Select a record set for EDA
main_rs_id = record_set_ids[0] if record_set_ids else None

if main_rs_id:
    df = dataframes[main_rs_id]
    print(f"Available columns in record set {main_rs_id}:")
    print(df.columns.tolist())

    # Try to pick a numeric field by heuristic (to handle anonymized @id column names)
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if not numeric_field and len(df.columns)>0:
        numeric_field = df.columns[0]  # fallback
    print(f"\nSelected numeric field for EDA (by @id): {numeric_field}")

    # Example filter: keep rows where value is greater than a threshold (median)
    if numeric_field:
        median_value = df[numeric_field].median() if pd.api.types.is_numeric_dtype(df[numeric_field]) else None
        threshold = median_value if median_value is not None else 0

        filtered_df = df[df[numeric_field]>threshold] if median_value is not None else df.copy()
        print(f"Filtered records with {numeric_field} > {threshold} (using @id):")
        display(filtered_df.head())

        # Normalization
        if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
            filtered_df[f"{numeric_field}_normalized"] = (
                filtered_df[numeric_field] - filtered_df[numeric_field].mean()
            ) / filtered_df[numeric_field].std()
            print(f"\nFirst 5 (de-identified) values for normalized '{numeric_field}' (by @id):")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to pick a group field (categorical)
        group_field = None
        for col in df.columns:
            # Pick a likely categorical (non-numeric) field
            if not pd.api.types.is_numeric_dtype(df[col]):
                group_field = col
                break
        if group_field:
            print(f"\nGrouping by {group_field} (by @id):")
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().to_frame("mean_").head()
            display(grouped)
        else:
            print("No categorical group field found for groupby.")
    else:
        print("No numeric field found.")
else:
    print("No record sets available.")

## 5. Visualization
Visualize distribution of the selected numeric field and its normalized values, as well as differences across groups, using matplotlib/seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field and pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of '{numeric_field}' (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Normalized plot
    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True, bins=15, color='g')
        plt.title(f"Normalized '{numeric_field}' distribution")
        plt.xlabel(f"{numeric_field}_normalized")
        plt.ylabel("Count")
        plt.show()

    # Boxplot by group
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"'{numeric_field}' grouped by '{group_field}' (@id)")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- Explored clinicopathological dataset in Croissant format using `mlcroissant`, referencing all entities by their `@id`.
- Identified record sets and available fields for extraction and EDA.
- Performed filtering, normalization, and group comparison using field `@id` references for maximum reproducibility and schema compliance.
- Visualized data distributions and group differences, providing a template for further analysis.